In [ ]:
# === Setup ===
# Runtime: <1 minute fast, <2 minutes full on a typical CPU (estimate).
# Hardware: CPU ok; no GPU required.
# Network: none; all datasets are generated locally.
# Competition-safe: general profile; check the actual contest package/data policy.
# Cẩm nang P08: NumPy, pandas, sklearn, Matplotlib, joblib; no package installation.
import os
import random
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
random.seed(42)
np.random.seed(42)
FAST = os.environ.get('OAI_FAST_MODE', '0') == '1'
rng = np.random.default_rng(42)
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)


# Debugging ML — Tìm lỗi bằng kiểm chứng nhỏ

Starter: baseline chạy hết; hoàn thành TODO trước khi đọc solution.

## Data → EDA

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

X = rng.normal(size=(160 if FAST else 400, 2))
y = (X[:, 0] - 0.7 * X[:, 1] + rng.normal(0, 0.15, len(X)) > 0).astype(int)
X_test = rng.normal(size=(64, 2))
test_ids = np.array([f'test_{i}' for i in range(len(X_test))])
tr, va = train_test_split(np.arange(len(y)), test_size=0.25, random_state=42, stratify=y)
assert set(tr).isdisjoint(va)
assert np.isfinite(X).all() and y.shape == (len(X),)
print('training shape:', X[tr].shape)
print('training class counts:', np.bincount(y[tr]))


## Preprocess → Model

Giữ float64 cho gradient check. Bias là phần tử cuối theta; không dùng validation để fit scaler.

In [ ]:
scaler = StandardScaler().fit(X[tr])
Xtr, Xva = scaler.transform(X[tr]), scaler.transform(X[va])
def probability(X, theta):
    """Map X (n,d), theta (d+1,) to binary probabilities (n,)."""
    z = X @ theta[:-1] + theta[-1]
    return np.exp(-np.logaddexp(0, -z))

def loss_and_grad(X, y, theta):
    """Return stable mean BCE scalar and gradient (d+1,) for y (n,)."""
    assert X.ndim == 2 and y.shape == (len(X),)
    assert theta.shape == (X.shape[1] + 1,)
    assert np.isfinite(X).all() and np.isfinite(theta).all()
    assert np.isin(y, [0, 1]).all()
    z = X @ theta[:-1] + theta[-1]
    loss = np.mean(np.logaddexp(0, z) - y * z)
    error = probability(X, theta) - y
    gradient = np.concatenate([X.T @ error / len(y), [error.mean()]])
    return float(loss), gradient

def fit_logistic(X, y, lr=0.1, epochs=300):
    """Return theta (d+1,) and epoch losses (epochs,) from zero initialization."""
    theta = np.zeros(X.shape[1] + 1, dtype=np.float64)
    history = []
    for _ in range(epochs):
        loss, gradient = loss_and_grad(X, y, theta)
        history.append(loss)
        # WHY: move opposite to the gradient to reduce loss locally.
        theta -= lr * gradient
    return theta, np.array(history)


## Worked Example → Diagnose

Tính tay g_w=-0.5, g_b=0 và một bước w_new=0.05. Dự đoán loss trước khi chạy.

In [ ]:
tiny_X = np.array([[-1.0], [1.0]])
tiny_y = np.array([0, 1])
tiny_theta = np.zeros(2)
initial_loss, g = loss_and_grad(tiny_X, tiny_y, tiny_theta)
assert np.allclose(initial_loss, np.log(2))
assert np.allclose(g, [-0.5, 0])
correct_step = tiny_theta - 0.1 * g
bad_step = tiny_theta + 0.1 * g
correct_loss, _ = loss_and_grad(tiny_X, tiny_y, correct_step)
bad_loss, _ = loss_and_grad(tiny_X, tiny_y, bad_step)
assert correct_loss < initial_loss < bad_loss
print('loss initial/correct/bad:', initial_loss, correct_loss, bad_loss)
# TODO: implement centered finite differences and diagnose a reversed gradient.
# TODO: add a tiny-overfit test and a label shape failure fixture.


## Train → Evaluate

**Hypothesis:** ba learning rate sẽ cho tốc độ giảm loss khác nhau. **Result:** loss curve và bảng F1. **Observation/Why:** ghi dựa vào số đo; loss giảm không thay thế validation.

In [ ]:
epochs = 200 if FAST else 500
rates = [0.1]
rows, candidates = [], {}
fig, ax = plt.subplots(figsize=(6, 3))
for lr in rates:
    theta, history = fit_logistic(Xtr, y[tr], lr=lr, epochs=epochs)
    pred = (probability(Xva, theta) >= 0.5).astype(int)
    score = f1_score(y[va], pred, labels=[0, 1], average='macro', zero_division=0)
    rows.append({'lr': lr, 'macro_f1': score, 'last_train_loss': history[-1]})
    candidates[lr] = theta
    ax.plot(history, label=f'lr={lr}')
    assert np.isfinite(history).all()
ax.set(title='Training loss by learning rate', xlabel='Epoch', ylabel='Mean BCE')
ax.legend()
plt.show()
plt.close(fig)
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
best_lr = sorted(rows, key=lambda r: (-r['macro_f1'], r['lr']))[0]['lr']


## Submit

Khóa lr, refit scaler và model trên toàn labeled data; tuyệt đối không fit trên test.

In [ ]:
final_scaler = StandardScaler().fit(X)
final_theta, _ = fit_logistic(final_scaler.transform(X), y, lr=best_lr, epochs=epochs)
test_pred = (probability(final_scaler.transform(X_test), final_theta) >= 0.5).astype(int)
np.savez(OUT / 'debug_model_starter.npz', theta=final_theta,
         mean=final_scaler.mean_, scale=final_scaler.scale_, lr=best_lr)


In [ ]:
# WHY: validate the file read from disk, not only the in-memory frame.
submission = pd.DataFrame({'id': test_ids, 'label': test_pred})
assert submission.columns.tolist() == ['id', 'label']
assert len(submission) == len(test_ids)
assert submission['id'].is_unique
assert submission['label'].isin([0, 1]).all()
submission.to_csv(OUT / 'submission_starter.csv', index=False)
reloaded = pd.read_csv(OUT / 'submission_starter.csv')
assert reloaded['id'].tolist() == list(test_ids)
assert reloaded['label'].tolist() == list(test_pred)
print('submission rows:', len(reloaded))


## Postmortem

Phân biệt ba evidence: gradient đúng trên fixture, batch dễ học được, validation F1. Ghi root cause và một trường hợp kiểm tra số có thể sai (điểm không khả vi hoặc precision thấp).